# GPT-3 — Language Models are Few-Shot Learners (2020)

_Papers · Transformers & LLMs_

**Scale a language model to 175 billion parameters and it learns new tasks from a few in-prompt examples — with no weight updates.**

---

This notebook builds a tiny in-context-learning demo step by step, then lets you watch accuracy climb as you add demonstrations to the prompt. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We illustrate **in-context learning** with a toy task: each input is a letter plus a stray symbol (e.g. `"a#"`) and the correct output drops the symbol (`"a"`). The "model" has **no weights to update** — it adapts only by reading `K` demonstration pairs from its context. We build this in three steps: (1) the task and its features, (2) the zero-shot vs. in-context predictors, (3) measuring accuracy as `K` grows.

### Step 1 — Define the toy task and a fixed feature map

We fix a small alphabet of letters and symbols. The `feat` function turns any string into a **bag-of-characters** count vector — it is fixed, *not* learned. `make_pair` draws a random (input, output) demonstration like `("a#", "a")`. This mirrors the flavor of the paper's Figure 1.2 task, "remove random symbols from a word".

In [ ]:
import numpy as np

# A TINY in-context-learning illustration -- OUR small run, NOT the paper's
# number. Hidden RULE (the model never sees it stated): an input is a letter
# followed by a stray symbol, e.g. "a#", and the correct output drops the
# symbol -> "a".
rng = np.random.default_rng(0)
letters = list("abcdefgh")
symbols = list("#@*")
alphabet = letters + symbols
idx = {ch: i for i, ch in enumerate(alphabet)}

def feat(s):
    # Fixed bag-of-characters map -- NOT learned. Counts each character.
    v = np.zeros(len(alphabet))
    for ch in s:
        v[idx[ch]] += 1.0
    return v

def make_pair():
    # One random demonstration: ("a#", "a").
    c = rng.choice(letters)
    s = rng.choice(symbols)
    return c + s, c

### Step 2 — Two predictors: zero-shot and in-context

With **no** demonstrations (`K = 0`), the frozen model has no example of the mapping, so it falls back to echoing the input — usually wrong. With one or more demonstrations (`K >= 1`), it finds the demonstration whose input is most similar (by feature dot-product), then applies the *same* deletion that demonstration shows. Crucially, no weights change in either case — the adaptation is pure reading of the context.

In [ ]:
def predict_zero_shot(query):
    # K = 0: no demonstrations. With no example of the mapping, the frozen
    # model falls back to echoing the input -> usually wrong. No weights involved.
    return query

def predict_in_context(query, demos):
    # K >= 1: find the demonstration whose INPUT is most similar, then apply
    # the SAME deletion it shows. Pure reading of the context -- no weight updates.
    qf = feat(query)
    sims = np.array([qf @ feat(di) for di, _ in demos])
    best = int(np.argmax(sims))
    di, do = demos[best]
    removed = set(di) - set(do)            # what that demo deleted
    return "".join(ch for ch in query if ch not in removed)

### Step 3 — Measure accuracy as the number of demonstrations K grows

`accuracy` evaluates a predictor on a fixed, separate eval set. We run it for the zero-shot case and for prefixes of a demonstration pool of increasing size `K`. The headline result: accuracy **climbs with K** even though no weights ever change — that is in-context learning. (Toy model, toy data: our illustration, not GPT-3's numbers.)

In [ ]:
def accuracy(demos, n=600):
    ev = np.random.default_rng(123)        # fixed eval set, separate from demos
    hits = 0
    for _ in range(n):
        c = ev.choice(letters)
        s = ev.choice(symbols)
        q, gold = c + s, c
        if demos is None:
            pred = predict_zero_shot(q)
        else:
            pred = predict_in_context(q, demos)
        hits += (pred == gold)
    return hits / n

pool = [make_pair() for _ in range(60)]    # demonstration pool; prompts are prefixes

print("zero-shot (K=0): %.3f" % accuracy(None))
print("one-shot  (K=1): %.3f" % accuracy(pool[:1]))
print("few-shot  (K=8): %.3f" % accuracy(pool[:8]))
print("\nin-context curve (K -> accuracy):")
for K in [0, 1, 2, 4, 8, 16]:
    a = accuracy(None) if K == 0 else accuracy(pool[:K])
    print("  K=%2d  acc=%.3f" % (K, a))

# zero-shot (K=0): 0.000
# one-shot  (K=1): 0.325
# few-shot  (K=8): 0.777
#
# in-context curve (K -> accuracy):
#   K= 0  acc=0.000
#   K= 1  acc=0.325
#   K= 2  acc=0.603
#   K= 4  acc=0.603
#   K= 8  acc=0.777
#   K=16  acc=0.952
# Accuracy CLIMBS with K, with NO weights changing -- that is in-context
# learning. Toy model, toy data: OUR illustration, NOT GPT-3's numbers.

## Visualize the data & results

_In-context learning means a FROZEN model (no weight updates) does a task by reading K examples in its prompt. If we put K demonstrations of a simple 'drop the symbol' rule in the context, does accuracy climb with K -- the shape of the paper's Figure 1.2?_

### Step 1 — Rebuild the task (self-contained for plotting)

This cell re-defines the toy task and the in-context predictor so the visualization runs on its own. It is the same logic as above — a frozen model that infers "drop the stray symbol" from `K` demonstrations — just gathered together so we can sweep `K` and chart the result.

In [ ]:
import numpy as np

# Toy IN-CONTEXT LEARNING -- OUR illustration, NOT the paper's number.
# A FROZEN model (no weights) infers 'drop the stray symbol' from K
# demonstrations in its context.
rng = np.random.default_rng(0)
letters, symbols = list("abcdefgh"), list("#@*")
alphabet = letters + symbols
idx = {ch: i for i, ch in enumerate(alphabet)}

def feat(s):
    v = np.zeros(len(alphabet))
    for ch in s:
        v[idx[ch]] += 1.0
    return v

def make_pair():
    c = rng.choice(letters)
    s = rng.choice(symbols)
    return c + s, c

def predict_in_context(query, demos):
    sims = np.array([feat(query) @ feat(di) for di, _ in demos])
    best = int(np.argmax(sims))
    di, do = demos[best]
    removed = set(di) - set(do)
    return "".join(ch for ch in query if ch not in removed)

### Step 2 — Sweep K and read the accuracy-vs-K curve

We evaluate accuracy on a fixed eval set for `K = 0, 1, 2, 4, 8, 16` and print the curve. The shape — accuracy rising as demonstrations accumulate, with weights frozen — is the point of Figure 1.2. (Our toy numbers, not GPT-3's benchmark results.)

In [ ]:
def accuracy(demos, n=600):
    ev = np.random.default_rng(123)
    hits = 0
    for _ in range(n):
        c = ev.choice(letters)
        s = ev.choice(symbols)
        q, gold = c + s, c
        pred = q if demos is None else predict_in_context(q, demos)  # K=0 echoes
        hits += (pred == gold)
    return hits / n

pool = [make_pair() for _ in range(60)]
for K in [0, 1, 2, 4, 8, 16]:
    a = accuracy(None) if K == 0 else accuracy(pool[:K])
    print("K=%2d  acc=%.3f" % (K, a))
# K= 0  acc=0.000
# K= 1  acc=0.325
# K= 2  acc=0.603
# K= 4  acc=0.603
# K= 8  acc=0.777
# K=16  acc=0.952
# Accuracy climbs with K, weights frozen -> in-context learning.
# OUR toy numbers, NOT GPT-3's benchmark results.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Classify the setting. For each prompt, say whether it is zero-shot, one-shot, or few-shot, and
            confirm whether any weights change. (a) "Translate to French: cheese =>". (b) "Translate to French:
            sea => mer. cheese =>". (c) "Translate to French: sea => mer. otter => loutre. cheese =>
            ... (eight pairs) ... cheese =>".

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the demonstrations $K$ in each prompt &mdash; the input-output pairs shown before the query. — _&sect;2 distinguishes the three settings only by $K$: 0, 1, or many._
- (a) has 0 demonstrations, (b) has 1, (c) has many (about 8). — _$K = 0$ is zero-shot, $K = 1$ is one-shot, $K$ in the tens is few-shot._
- Check weights: in all three, the model is only reading text &mdash; no gradient updates. — _The abstract: GPT-3 is applied "without any gradient updates or fine-tuning" in every setting._

**Answer:** (a) Zero-shot ($K = 0$). (b) One-shot ($K = 1$). (c) Few-shot ($K \approx 8$).
                 In all three, no weights change &mdash; the demonstrations are conditioning text, read
                 in the forward pass. That is in-context learning, not training.

</details>

**Problem 2.** Why is it called "learning" if nothing is learned? A skeptic says: "If GPT-3's weights never
            change, then it is not learning the task &mdash; calling it a few-shot learner is marketing."
            Answer the skeptic using the paper's framing.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- State where the adaptation happens: in the forward pass over the prompt, not in the weights. — _In-context learning (&sect;1) is the model "rapidly adapting to or recognizing the desired task" at inference time._
- Explain the mechanism: an autoregressive model conditions its next-token probability on the whole prompt, so demonstrations reshape the prediction without reshaping the weights. — _Continuing a demonstrated input-output pattern is just high-probability next-token prediction for a model trained on patterned text._
- Concede the precise sense: nothing is learned permanently; a different prompt next turn gets handled by the same frozen weights. — _That impermanence is exactly why no gradient updates are needed &mdash; the "learning" lives in the context._

**Answer:** The skeptic is half right: nothing is learned permanently &mdash; the weights are frozen,
                 and the next prompt is handled by the same parameters. But "learning" here refers to the model's
                 in-context adaptation: by conditioning its next-token distribution on the demonstrations, it
                 recognizes and continues the task's pattern within a single forward pass. The paper (&sect;1) names
                 this "in-context learning." It is genuine task adaptation; it just lives in the prompt, not the
                 weights &mdash; which is the whole point of "no gradient updates."

</details>

**Problem 3.** Ablation &mdash; remove the scale. Figure 1.2 shows larger models have steeper accuracy-vs-K
            curves. Suppose you re-ran the same in-context task on a much smaller model. Predict how the
            curve would change, and what that implies about whether in-context learning is "free."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the figure's claim: "Larger models make increasingly efficient use of in-context information," with steeper curves for larger models. — _The benefit of each added demonstration grows with model size &mdash; that is the measured trend._
- Predict the small-model curve: flatter &mdash; accuracy rises little (or not at all) as K grows, because the small model is worse at recognizing the pattern from context. — _A flatter accuracy-vs-K slope is exactly what "less efficient use of in-context information" means._
- Draw the implication: in-context learning is an ability that emerges with scale, not a property every model has. — _If it were free, small and large models would have the same slope &mdash; but they do not._

**Answer:** On a much smaller model the accuracy-vs-K curve would be flatter: adding demonstrations
                 would help little, because the small model is weaker at recognizing and continuing the in-context
                 pattern. The implication is the paper's thesis &mdash; in-context learning is not free; it
                 strengthens with scale. "Larger models make increasingly efficient use of in-context
                 information" (Figure 1.2), so the steep, useful few-shot curve is itself a product of size.

</details>